In [1]:
import json
import random
import subprocess
import time
import itertools
import os
from collections import Counter
import pandas as pd
import shutil


## Read in Data and Clean

In [4]:
with open("oracle-cards-20260508090243.json", "r", encoding="utf-8") as file:
    FullCardList = json.load(file)

# remove add duplicates explained below ======================================================================
before = len(FullCardList)
print("Total Before Removal: ", before)

itemsRemoved = {}
illegal = []

for c in FullCardList[:]: # copy so we can delete and iterate at same time
    if "//" in c["name"]:
        parts=c["name"].split("//")
        if parts[0].strip() == parts[1].strip():
            itemsRemoved[c["name"]] = c.items()
            FullCardList.remove(c)

# remove non-legal standard cards ============================================================================
for c in FullCardList[:]:
    if ((c["legalities"]["standard"] == "not_legal") ):
        if c["name"] not in itemsRemoved:
            illegal.append(c["name"])
            FullCardList.remove(c)


print("number of items removed (//): ", len(itemsRemoved))
print("number of non-legal removed: ", len(illegal))

print("")
print("Total after removal: ", len(FullCardList))
print("Total number of items removed: ", before - (len(FullCardList)))

Total Before Removal:  37442
number of items removed (//):  2189
number of non-legal removed:  30810

Total after removal:  4440
Total number of items removed:  33002


In [5]:
#for items in itemsRemoved:
 #   print(items)
'''
-non-legal cards that should be removed, these are labelled as legal
listed as dual sided cards however base printings are single sided-
'''

'\n-non-legal cards that should be removed, these are labelled as legal\nlisted as dual sided cards however base printings are single sided-\n'

In [6]:
#for items in illegal:
 #   print(items)

In [7]:
'''
seeing what card elements to keep
following where seen as irrelevant
'''

removeElements = ["set_type",
                  "mtgo_id",
                  "tcgplayer_id",
                  "lang",
                  "released_at",
                  "uri",
                  "scryfall_uri",
                  "highres_image",
                  "image_status",
                  "image_uris",
                  "set_uri",
                  "set_search_uri",
                  "scryfall_set_uri",
                  "rulings_uri",
                  "prints_search_uri",
                  "watermark",
                  "artist",
                  "artist_ids",
                  "illustration_id",
                  "border_color",
                  "frame",
                  "frame_effects",
                  "security_stamp",
                  "full_art",
                  "testless",
                  "booster",
                  "story_spotlight",
                  "prices",
                  "related_uris",
                  "purchase_uris",
                  "digital",
                  "foil",
                  "nonfoil",
                  "preview",
                  "games",
                  "finishes",
                  "oversized",
                  "promo",
                  "reprint",
                  "textless"]

for card in FullCardList:
    for key in removeElements:
        card.pop(key,None)
                  

In [8]:
cards = {card["name"]: card for card in FullCardList}

### List of all standard legal cards

In [9]:
cards = dict(sorted(cards.items()))
for key, value in cards.items():
    print(key, " :", value["type_line"])

A Killer Among Us  : Enchantment
A Realm Reborn  : Enchantment
A Tale for the Ages  : Enchantment
Aang's Iceberg  : Enchantment
Aang's Journey  : Sorcery — Lesson
Aang, Swift Savior // Aang and La, Ocean's Fury  : Legendary Creature — Human Avatar Ally // Legendary Creature — Avatar Spirit Ally
Aang, at the Crossroads // Aang, Destined Savior  : Legendary Creature — Human Avatar Ally // Legendary Creature — Avatar Ally
Aang, the Last Airbender  : Legendary Creature — Human Avatar Ally
Aatchik, Emerald Radian  : Legendary Creature — Insect Druid
Abandon Attachments  : Instant — Lesson
Abandoned Air Temple  : Land
Abandoned Campground  : Land
Aberrant Manawurm  : Creature — Wurm
Abhorrent Oculus  : Creature — Eye
Abigale, Eloquent First-Year  : Legendary Creature — Bird Bard
Abigale, Poet Laureate // Heroic Stanza  : Legendary Creature — Bird Bard // Sorcery
Abrade  : Instant
Abraded Bluffs  : Land — Desert
Absolute Virtue  : Legendary Creature — Avatar Warrior
Absolving Lammasu  : Creat

### Colour specific card lists

In [11]:
#
##
# here card lists are made for all mono colour and all dual colour cards lists, 
# the names chosen are the commonly used name by the community 
#
#
blackCards = {}
blueCards = {}
greenCards = {}
redCards = {}
whiteCards = {}

azoriousCards = {} #white blue 1
dimirCards = {} # blue black 2
rakdosCards = {} # black red 3
gruulCards = {} # red green 4
selesnyaCards = {} #green white 5

orzhovCards = {} # white black 6
golgariCards = {} #black green 7
simicCards = {} #green blue 8
izzetCards = {} #blue red 9
borosCards = {} #red white 10


multiColourCards = {}

for key, value in cards.items():

#all mono colour cards
    #print(value["color_identity"])
    if (value["color_identity"] == ["B"] ):
        #print(key)
        blackCards[key] = value

    elif (value["color_identity"] == ["U"] ):
        blueCards[key] = value

    elif (value["color_identity"] == ["G"] ):
        greenCards[key] = value

    elif (value["color_identity"] == ["R"] ):
        redCards[key] = value

    elif (value["color_identity"] == ["W"] ):
        whiteCards[key] = value
    
#multi colours just for keeping track
    else:
        multiColourCards[key] = value

# many decks use more than on colour
# two colour combinations
    if (set(value["color_identity"]).issubset(["U","W"])): #1
        azoriousCards[key] = value

    if (set(value["color_identity"]).issubset(["B","U"])): #2
        dimirCards[key] = value

    if (set(value["color_identity"]).issubset(["B","R"])): #3
        rakdosCards[key] = value

    if (set(value["color_identity"]).issubset(["G","R"])): #4
        gruulCards[key] = value

    if (set(value["color_identity"]).issubset(["G","W"])): #5
        selesnyaCards[key] = value


    if (set(value["color_identity"]).issubset(["B","W"])): #6
        orzhovCards[key] = value

    if (set(value["color_identity"]).issubset(["B","G"])): #7
        golgariCards[key] = value

    if (set(value["color_identity"]).issubset(["U","G"])): #8
        simicCards[key] = value

    if (set(value["color_identity"]).issubset(["U","R"])): #9
        izzetCards[key] = value

    if (set(value["color_identity"]).issubset(["W","R"])): #10
        borosCards[key] = value


print("azorious Cards:    ", len(azoriousCards)) #1
print("dimir Cards:       ", len(dimirCards)) #2
print("rakdos Cards:      ", len(rakdosCards)) #3
print("gruul Cards:       ", len(gruulCards)) #4
print("selesnya Cards:    ", len(selesnyaCards)) #5
print("orzhov Cards:      ", len(orzhovCards)) #6
print("golgari Cards:     ", len(golgariCards)) #7
print("simic Cards:       ", len(simicCards)) #8
print("izzet Cards:       ", len(izzetCards)) #9
print("boros Cards:       ", len(borosCards)) #10


Alldicts = (blackCards, blueCards, greenCards, redCards ,whiteCards ,azoriousCards ,dimirCards ,rakdosCards, gruulCards, selesnyaCards, orzhovCards, golgariCards, simicCards,izzetCards ,borosCards )

azorious Cards:     1649
dimir Cards:        1655
rakdos Cards:       1668
gruul Cards:        1640
selesnya Cards:     1648
orzhov Cards:       1691
golgari Cards:      1680
simic Cards:        1655
izzet Cards:        1677
boros Cards:        1683


In [194]:
len(blackCards) + len(whiteCards) + len(greenCards) + len(redCards) + len(blueCards) + len(multiColourCards)

4440

## Searching

### check if card name contains "(text)" 

In [184]:

search = "orzhov"

for c in FullCardList:
    if search.lower() in c["name"].lower():
        print(c["name"])

Orzhov Guildgate


### layout types

In [36]:
search = ""
layoutTypes = set()

for c in FullCardList:
    if search in c["name"]:
        layoutTypes.add(c["layout"])

print(layoutTypes)

{'normal', 'transform', 'saga', 'modal_dfc', 'meld', 'class', 'prepare', 'adventure', 'case', 'split'}


In [37]:
search = "//"
layoutTypes = set()

for c in FullCardList:
    if search in c["name"]:
        layoutTypes.add(c["layout"])

print(layoutTypes)

{'normal', 'transform', 'modal_dfc', 'adventure', 'split', 'prepare'}


### Land list

In [24]:
#lands

search = "Land"
landcount = 0

for c in FullCardList:
    if search in c["type_line"]:
        print(c["type_line"])
        landcount +=1

Land
Land
Land
Land — Cave
Land — Swamp Mountain
Basic Land
Land
Land — Island Swamp
Land
Artifact // Land
Land
Land
Land
Land — Town
Land — Mountain Forest
Legendary Artifact // Legendary Land
Land — Island Mountain
Land
Land
Land
Land — Cave
Land
Land
Legendary Artifact // Legendary Artifact Land
Land — Forest Island
Land — Plains Swamp
Land
Land — Town
Land
Land — Town
Land
Land
Land — Town
Land
Land
Land — Cave
Artifact // Land — Cave
Land
Land
Legendary Artifact // Legendary Artifact Land
Land
Land
Land
Land
Land
Legendary Creature — God // Land
Land
Land — Gate
Land — Desert
Land
Land — Desert
Land
Land
Land
Land — Desert
Land
Land
Land
Land
Land — Swamp Mountain
Land — Mountain Plains
Land
Land
Land
Land
Land
Legendary Creature — God // Land
Land
Land
Land
Land — Cave
Land — Town // Instant — Adventure
Land
Land
Land — Town // Sorcery — Adventure
Land
Land — Town // Sorcery — Adventure
Land — Planet
Land — Cave
Land — Planet
Land
Enchantment // Land — Cave
Land — Gate
Land
Land 

In [25]:
landcount

264

### Specific search (name + layout)

In [38]:

search = "//"
search2 = "prepare"

for c in FullCardList:
    if search.lower() in c["name"].lower():
        if search2.lower() in c["layout"].lower():
            print(c["name"], " : ", c["layout"])
            

Adventurous Eater // Have a Bite  :  prepare
Maelstrom Artisan // Rocket Volley  :  prepare
Blazing Firesinger // Seething Song  :  prepare
Goblin Glasswright // Craft with Pride  :  prepare
Emeritus of Ideation // Ancestral Recall  :  prepare
Scathing Shadelock // Venomous Words  :  prepare
Abigale, Poet Laureate // Heroic Stanza  :  prepare
Vastlands Scavenger // Bind to Life  :  prepare
Scheming Silvertongue // Sign in Blood  :  prepare
Infirmary Healer // Stream of Life  :  prepare
Studious First-Year // Rampant Growth  :  prepare
Emeritus of Conflict // Lightning Bolt  :  prepare
Landscape Painter // Vibrant Idea  :  prepare
Honorbound Page // Forum's Favor  :  prepare
Grave Researcher // Reanimate  :  prepare
Tam, Observant Sequencer // Deep Sight  :  prepare
Skycoach Conductor // All Aboard  :  prepare
Leech Collector // Bloodletting  :  prepare
Emeritus of Woe // Demonic Tutor  :  prepare
Campus Composer // Aqueous Aria  :  prepare
Sanar, Unfinished Genius // Wild Idea  :  prep

### Specific Search (name + info)

In [207]:
search = "Fire Lord Zuko"
for key, value in cards[search].items():
    print(key, " : ", value)

object  :  card
id  :  e62d3bcc-7bb4-42be-90a9-caf3c1caa29d
oracle_id  :  a129423d-c886-4120-96f7-c2517ea7eb83
multiverse_ids  :  []
resource_id  :  FC5D57B484A213622B34B8F44F5113EB1D38784C0E2DC82C88517394E1B0EFC1
cardmarket_id  :  842857
name  :  Fire Lord Zuko
layout  :  normal
mana_cost  :  {R}{W}{B}
cmc  :  3.0
type_line  :  Legendary Creature — Human Noble Ally
oracle_text  :  Firebending X, where X is Fire Lord Zuko's power. (Whenever this creature attacks, add X {R}. This mana lasts until end of combat.)
Whenever you cast a spell from exile and whenever a permanent you control enters from exile, put a +1/+1 counter on each creature you control.
power  :  2
toughness  :  4
colors  :  ['B', 'R', 'W']
color_identity  :  ['B', 'R', 'W']
keywords  :  ['Firebending']
produced_mana  :  ['R']
legalities  :  {'standard': 'legal', 'future': 'legal', 'historic': 'legal', 'timeless': 'legal', 'gladiator': 'legal', 'pioneer': 'legal', 'modern': 'legal', 'legacy': 'legal', 'pauper': 'not_lega

In [40]:
search = "Plains"

for key, value in cards[search].items():
    print(key, " : ", value)

object  :  card
id  :  24dc369c-020a-4115-a4bb-d60a44de64e3
oracle_id  :  bc71ebf6-2056-41f7-be35-b2e5c34afa99
multiverse_ids  :  []
name  :  Plains
layout  :  normal
mana_cost  :  
cmc  :  0.0
type_line  :  Basic Land — Plains
oracle_text  :  ({T}: Add {W}.)
colors  :  []
color_identity  :  ['W']
keywords  :  []
produced_mana  :  ['W']
legalities  :  {'standard': 'legal', 'future': 'legal', 'historic': 'legal', 'timeless': 'legal', 'gladiator': 'legal', 'pioneer': 'legal', 'modern': 'legal', 'legacy': 'legal', 'pauper': 'legal', 'vintage': 'legal', 'penny': 'legal', 'commander': 'legal', 'oathbreaker': 'legal', 'standardbrawl': 'legal', 'brawl': 'legal', 'alchemy': 'legal', 'paupercommander': 'legal', 'duel': 'legal', 'oldschool': 'not_legal', 'premodern': 'legal', 'predh': 'legal', 'tlr': 'legal'}
reserved  :  False
game_changer  :  False
variation  :  False
set_id  :  59218c23-cb7a-4fde-9cff-3f2f71af1308
set  :  hob
set_name  :  The Hobbit
collector_number  :  194
rarity  :  common


### Specific search (typeline)

In [41]:
search = "basic"

for c in FullCardList:
    if search.lower() in c["type_line"].lower():
        print(c["name"] , " : ", c["type_line"])

Wastes  :  Basic Land
Swamp  :  Basic Land — Swamp
Mountain  :  Basic Land — Mountain
Island  :  Basic Land — Island
Forest  :  Basic Land — Forest
Plains  :  Basic Land — Plains


## Functions

In [49]:
##
# cards is the dictionary working with
# cards and their info
#

print("   \"cards\" is a dictionary of dictionaries")
print("Type of \"cards\"                          :", type(cards))
print("Types of items in \"cards\"                :", type(cards["Plains"]))
print("length of \"cards\"                        :", len(cards))
print("")

print("   \"FullCardList\" is a List of dictionaries")
print("Type of \"FullCardList\"                   :", type(FullCardList))
print("Types of items in \"FullCardList\"         :", type(FullCardList[1]))
print("length of \"FullCardList\"                 :", len(cards))

   "cards" is a dictionary of dictionaries
Type of "cards"                          : <class 'dict'>
Types of items in "cards"                : <class 'dict'>
length of "cards"                        : 4440

   "FullCardList" is a List of dictionaries
Type of "FullCardList"                   : <class 'list'>
Types of items in "FullCardList"         : <class 'dict'>
length of "FullCardList"                 : 4440


### Writing to File

In [2]:
'''
make sure all cards have a name
set
and collector number
'''


def writeToFile(name, cardList, decklist, verbose=False):

    path = "decklists/" + name + ".dck"

    with open(path, "w") as file:
        file.write("[metadata]\n")
        file.write("Name=" + name + "\n")
        file.write("[Main]\n")

        for name in decklist:

            if( (cardList[name]["layout"] == "normal") or 
               (cardList[name]["layout"] == "saga") or 
               (cardList[name]["layout"] == "split") or
               (cardList[name]["layout"] == "meld") or
               (cardList[name]["layout"] == "class") ):
                write = "1 " + name + "|" + cardList[name]["set"] + "|" + cardList[name]["collector_number"] + "\n"
                file.write(write)

            elif( (cardList[name]["layout"] == "transform") or 
                 (cardList[name]["layout"] == "flip") or 
                 (cardList[name]["layout"] == "prepare") or
                 (cardList[name]["layout"] == "adventure") or
                 (cardList[name]["layout"] == "modal_dfc") ):
                parts = name.split("//")
                write = "1 " + parts[0].strip() + "|" + cardList[name]["set"] + "|" + cardList[name]["collector_number"] + "\n"
                file.write(write)

            if verbose:
                print(write)
            

### Check Legality

In [114]:
## put in main loop
def isLegal(length, cardlist, decklist):
    if len(decklist) != length:
        return False

    for name, count in Counter(decklist).items():

        if "basic land" in cardlist[name]["type_line"].lower():
            continue
        if count > 4:
            return False

    return True

### Evaluate Deck

In [3]:

def evalDeck(name, cardList, deckList, verbose = False):

    score = 0

    # write to .dck file
    writeToFile(name, cardList, deckList, verbose)

    #play agisnt decks
    #get results

    #return score

### Generate Random Decks

In [68]:

##
# this will generate a random decklist one card at atime allowing cards to be added multiple times
#
#

def randDeck(cardlist, deckSize, verbose=False):
    rOutput = []
    for n in range(deckSize):
        selectR = random.sample(list(cardlist.keys()), 1)
        count = rOutput.count(selectR)

        if "basic land" in cardlist[selectR[0]]["type_line"].lower():
            rOutput.append(selectR[0])

        elif count < 4:
            rOutput.append(selectR[0])

    if verbose:
        print(rOutput)
        print( len(rOutput))
    return rOutput

    
##
# card games heavily lean towards containing multiples of cards so next random generation method willatt 2 of each card  
#
#
def randDeck2(cardlist, deckSize, verbose = False):
    rOutput = []
    for n in range(deckSize // 2):
        selectR = random.sample(list(cardlist.keys()), 1)
        count = rOutput.count(selectR)

        if "basic land" in cardlist[selectR[0]]["type_line"].lower():
            rOutput.append(selectR[0])
            rOutput.append(selectR[0])

        elif count < 4:
            rOutput.append(selectR[0])
            rOutput.append(selectR[0])
            
    if verbose:
        print(rOutput)
        print( len(rOutput))
    return rOutput  

##
#
#
#
def randDeckMulti(cardlist, deckSize, verbose = False):
    rOutput = []
    size = 0
    while size < deckSize:
        selectR = random.sample(list(cardlist.keys()), 1)
        count = rOutput.count(selectR)

        numCopies = random.randint(1,4)

        numCopies = numCopies - count

        for n in range(numCopies):
            if "basic land" in cardlist[selectR[0]]["type_line"].lower():
                rOutput.append(selectR[0])
                rOutput.append(selectR[0])

            elif count < 4:
                rOutput.append(selectR[0])

        size = len(rOutput)

    rOutput = rOutput[:deckSize]

    if verbose:
        print(rOutput)
        print( len(rOutput))
    return rOutput
            

### Crossover

In [1]:
def crossoverPositonal(deck1, deck2, verbose= False):
    if len(deck1) != len(deck2):
        print("diff len")
        return none
    
    newDeck =[]
    for n in range(len(deck1)):
        temp = random.sample([deck1[n],deck2[n]], 1)
        newDeck.append(temp[0])

    if verbose:
        print(newDeck)
        print( len(newDeck))
    return newDeck

def crossoverPoint(deck1, deck2, verbose= False):
    if len(deck1) != len(deck2):
        print("diff len")
        return none

    
    point = random.randint(1, len(deck1) -1)
    newDeck = deck1[:point] + deck2[point:]
    if verbose:
        print(newDeck)
        print( len(newDeck))
    return newDeck

### Mutation

In [107]:
##
# goes through every spot in the deck and with mutaterate probablity it mutates it into a random card
#
#

def mutateSpot(deck, cardlist, mutateRate, verbose=True):
    newDeck = deck[:]

    for n in range(len(deck)):
        if random.random() < mutateRate:
            selectR = random.sample(list(cardlist.keys()), 1)
            while(selectR[0] in deck or selectR[0] in newDeck):
                selectR = random.sample(list(cardlist.keys()), 1)
            newDeck[n] = selectR[0]

    if verbose:
        print(newDeck)
        print( len(newDeck))
    return newDeck
    

##
# 
#
#
def mutateCard(deck, cardlist, mutateRate, verbose=True):
    tempDeck = deck[:]
    arr = set(deck)
    for card in arr:
        if "basic land" in cardlist[card]["type_line"].lower():
            continue
        if random.random() < mutateRate:
            selectR = random.sample(list(cardlist.keys()), 1)
            while(selectR[0] in deck or selectR[0] in tempDeck):
                selectR = random.sample(list(cardlist.keys()), 1)

            tempDeck =  [x for x in tempDeck if x !=card]
            for n in range(len(deck) - len(tempDeck)):
                tempDeck.append(selectR[0])

        newDeck = tempDeck[:len(deck)]

    if verbose:
        print(newDeck)
        print( len(newDeck))
    return newDeck


##
#
#
#
def mutateCopy(deck, cardlist, mutateRate, verbose=True):
    newDeck = deck[:]
    for i, card in enumerate(newDeck):
        if random.random() < mutateRate:
            selectR = random.choice(deck)
            if (newDeck.count(selectR) < 4):
                newDeck[i] = selectR
    if verbose:
        print(newDeck)
        print( len(newDeck))
    return newDeck

### Main Body Funciton

In [12]:
##
# populaiton is a list of lists which contain a list (the deck) and a score (fitness)
# [
#  [deck, score]
# ]
# deck = [mountain, plains, ....]
#
def mainGeneticAlgo(
    #fitness Funciton
        f,
    #check legality function
    #inputs (length, cardlist, decklist)
        legal,
    #write to file function
    #inputs = (name, cardList, decklist, verbose=False):
        write,
    
    #init function
    #inputs = (cardlist, deckSize, verbose=False)
        init,
    #crossover
    # inputs = (deck1, deck2, verbose= False)
        crossover,
    #mutate
    #inputs = (deck, cardlist, mutateRate, verbose=True):
        mutate,

    #All dictionaries, this allows for generation of specfically coloured decks
        allDictionaries,
    #boolean to tell if to use colour specific dictionaries or not
        useAllDicts,
    #card list
        cardsList,
    #population size
        popSize = 16,
    #number of generations
        generations = 100,
    #deck size
        deckSize = 60    
    #12 inputs
):
#=========================================================
    currentgen = 0
    going = True

    ##init --------------------------------------
    population = []
    while len(population) < popSize:
        createGoing = True
        while(createGoing):
            temp = init(cardsList, deckSize, False)

            if legal(deckSize, cardsList, temp):
                createGoing = False
        population.append([temp,0])

    while(going):
        currentgen +=1

    ##fitness


    ##selection


    ##crossover


    ##mutation


    ##add some more randomness


    ##check legality

        if(currentgen > generations):
            going = False

    return population

In [138]:
geneticAlgo("na", isLegal, writeToFile, randDeck, crossoverPoint, mutateCopy, cards, 16, 1)

[[['Walltop Sentries',
   'Scampering Surveyor',
   'Pest Control',
   'Meat Locker // Drowned Diner',
   'Angel of Finality',
   'Lionheart Glimmer',
   'Scalestorm Summoner',
   'Raucous Audience',
   'Balamb Garden, SeeD Academy // Balamb Garden, Airborne',
   'Elixir',
   'Claim Jumper',
   'Concealed Courtyard',
   'Rakdos Joins Up',
   'Dire Downdraft',
   'Gongaga, Reactor Town',
   'Arahbo, the First Fang',
   'Wilt in the Heat',
   'Rinoa Heartilly',
   'Soaring Sandwing',
   'Goobbue Gardener',
   'Group Project',
   'Hellish Sideswipe',
   'Voyager Glidecar',
   'Rakdos, the Muscle',
   'Lys Alana Informant',
   'Lavaspur Boots',
   'Unravel',
   'Eclipsed Kithkin',
   'Coruscation Mage',
   'Jazal Goldmane',
   'Might of the Meek',
   'Archangel of Tithes',
   'Spider-Woman, Stunning Savior',
   'Gene Pollinator',
   'Dalkovan Packbeasts',
   'Mild-Mannered Librarian',
   'Poison Dart Frog',
   'Corrupted Conviction',
   'The Last Ride',
   'Bot Bashing Time',
   'Dawn-Bles